# Notebook 04 — SAE Dilution on the mod30 Residue Manifold

This notebook tests whether a top-k sparse autoencoder (SAE) compactly recovers the same 8-lane mod30 residue manifold recovered by NMF in Notebook 03.

**Notebook chain**

- Notebook 01: define the mod30 residue manifold.
- Notebook 02: show constrained sampling improves signal access.
- Notebook 03: show NMF recovers residue-lane structure.
- Notebook 04: show SAE dictionaries can fragment capacity into dead, redundant, or diffuse features.

**Core claim**

> In this controlled residue-manifold setting, a top-k sparse autoencoder may exhibit dilution: extra dictionary capacity does not automatically translate into clean manifold coverage.


## 1. Setup

Figures are saved as SVG only. No PNG duplicates.


In [ ]:
import os
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl

import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.decomposition import NMF
from sklearn.metrics import mean_squared_error

mpl.rcParams["svg.fonttype"] = "none"
mpl.rcParams["figure.dpi"] = 120

MOD = 30
VALID_LANES_MOD30 = [1, 7, 11, 13, 17, 19, 23, 29]

Path("data").mkdir(exist_ok=True)
Path("figures").mkdir(exist_ok=True)


def save_svg(fig, name):
    path = f"figures/{name}.svg"
    fig.savefig(path, bbox_inches="tight")
    print(f"Saved: {path}")

print("Ready.")

## 2. Generate constrained residue-count matrix

Each row is a sampled batch. Each column is one residue class mod30.

This matches Notebook 03 so SAE and NMF comparisons are grounded in the same controlled setup.


In [ ]:
def make_batch_matrix(
    n_batches=600,
    batch_size=250,
    constrained=True,
    seed=9423,
):
    rng = np.random.default_rng(seed)
    all_numbers = np.arange(2, 100_000)
    valid_numbers = all_numbers[np.isin(all_numbers % MOD, VALID_LANES_MOD30)]

    rows = []
    for _ in range(n_batches):
        sample_space = valid_numbers if constrained else all_numbers
        sample = rng.choice(sample_space, size=batch_size, replace=True)
        counts = np.bincount(sample % MOD, minlength=MOD).astype(np.float32)
        rows.append(counts)

    X = np.array(rows, dtype=np.float32)
    X = X / X.sum(axis=1, keepdims=True)
    return X

X = make_batch_matrix()
print("X shape:", X.shape)
print("Active residue columns:", np.where(X.sum(axis=0) > 0)[0].tolist())

## 3. Define a top-k sparse autoencoder

This is a deliberately small, transparent SAE-style model:

- encoder: residue vector → dictionary activations
- top-k selection: only k features survive per sample
- decoder: selected features → residue reconstruction

The goal is not to optimize a production SAE. The goal is to test whether this sparse dictionary reliably recovers the known 8-lane manifold.


In [ ]:
class TopKSAE(nn.Module):
    def __init__(self, input_dim=30, hidden_dim=16, topk=2):
        super().__init__()
        self.encoder = nn.Linear(input_dim, hidden_dim)
        self.decoder = nn.Linear(hidden_dim, input_dim, bias=False)
        self.topk = topk

    def topk_activation(self, z):
        z_relu = F.relu(z)
        values, indices = torch.topk(z_relu, self.topk, dim=1)
        mask = torch.zeros_like(z_relu)
        mask.scatter_(1, indices, 1.0)
        return z_relu * mask

    def forward(self, x):
        z = self.encoder(x)
        z_sparse = self.topk_activation(z)
        x_hat = self.decoder(z_sparse)
        x_hat = F.relu(x_hat)
        return x_hat, z_sparse


def train_sae(
    X,
    hidden_dim=16,
    topk=2,
    epochs=1000,
    lr=1e-2,
    seed=9423,
):
    torch.manual_seed(seed)
    np.random.seed(seed)

    model = TopKSAE(input_dim=X.shape[1], hidden_dim=hidden_dim, topk=topk)
    x = torch.tensor(X, dtype=torch.float32)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    losses = []

    for epoch in range(epochs):
        opt.zero_grad()
        x_hat, z = model(x)
        loss = F.mse_loss(x_hat, x)
        loss.backward()
        opt.step()
        losses.append(loss.item())

    with torch.no_grad():
        x_hat, z = model(x)
        final_loss = F.mse_loss(x_hat, x).item()

    decoder = model.decoder.weight.detach().cpu().numpy().T
    decoder = np.maximum(decoder, 0)
    activations = z.detach().cpu().numpy()
    x_hat_np = x_hat.detach().cpu().numpy()

    return model, decoder, activations, losses, x_hat_np, final_loss

print("TopKSAE defined.")

## 4. Feature diagnostics

We measure each dictionary feature by:

- **peak residue**: where the decoder feature is largest
- **lane mass ratio**: feature mass located on valid mod30 lanes
- **activation count**: how often the feature is used
- **dead feature**: activation count equals zero
- **coverage**: fraction of the 8 valid lanes represented by at least one feature
- **redundancy**: multiple features peaking on the same valid lane


In [ ]:
lane_mask = np.zeros(MOD, dtype=bool)
lane_mask[VALID_LANES_MOD30] = True


def feature_lane_mass(feature):
    total = feature.sum()
    if total <= 1e-12:
        return 0.0
    return float(feature[lane_mask].sum() / total)


def feature_peak_residue(feature):
    if feature.sum() <= 1e-12:
        return -1
    return int(np.argmax(feature))


def summarize_features(decoder, activations, activation_threshold=1e-8):
    rows = []
    active_counts = (activations > activation_threshold).sum(axis=0)

    for j, feature in enumerate(decoder):
        feature = np.maximum(feature, 0)
        peak = feature_peak_residue(feature)
        rows.append({
            "feature": j,
            "peak_residue": peak,
            "lane_mass_ratio": feature_lane_mass(feature),
            "activation_count": int(active_counts[j]),
            "dead": int(active_counts[j] == 0),
            "valid_peak": int(peak in VALID_LANES_MOD30),
        })

    df = pd.DataFrame(rows)
    valid_peaks = df[df["valid_peak"] == 1]["peak_residue"].tolist()
    unique_valid_peaks = sorted(set(valid_peaks))

    coverage = len(unique_valid_peaks) / len(VALID_LANES_MOD30)
    redundancy = len(valid_peaks) - len(unique_valid_peaks)
    dead_features = int(df["dead"].sum())

    summary = {
        "coverage": coverage,
        "unique_valid_lanes": len(unique_valid_peaks),
        "redundant_valid_features": redundancy,
        "dead_features": dead_features,
        "mean_lane_mass_ratio": float(df["lane_mass_ratio"].mean()),
        "median_lane_mass_ratio": float(df["lane_mass_ratio"].median()),
    }

    return df, summary

print("Diagnostics ready.")

## 5. Sweep dictionary size and top-k

This sweep tests whether more capacity automatically yields better lane recovery.


In [ ]:
dictionary_sizes = [8, 12, 16, 24, 32]
topks = [1, 2, 4]

records = []
feature_tables = []

for hidden_dim in dictionary_sizes:
    for topk in topks:
        model, decoder, activations, losses, x_hat, final_loss = train_sae(
            X,
            hidden_dim=hidden_dim,
            topk=topk,
            epochs=800,
            lr=1e-2,
            seed=9423 + hidden_dim * 10 + topk,
        )

        df_feat, summary = summarize_features(decoder, activations)
        summary.update({
            "hidden_dim": hidden_dim,
            "topk": topk,
            "final_loss": final_loss,
            "initial_loss": losses[0],
            "loss_drop": losses[0] - final_loss,
        })
        records.append(summary)

        df_feat["hidden_dim"] = hidden_dim
        df_feat["topk"] = topk
        feature_tables.append(df_feat)

        print(f"hidden={hidden_dim:2d}, topk={topk}: coverage={summary['coverage']:.3f}, dead={summary['dead_features']}, redundant={summary['redundant_valid_features']}, loss={final_loss:.6g}")

df_summary = pd.DataFrame(records)
df_features = pd.concat(feature_tables, ignore_index=True)

df_summary

## 6. Representative SAE dictionary

We use `hidden_dim = 16`, `topk = 2` as a representative capacity-above-ground-truth case.


In [ ]:
REP_HIDDEN = 16
REP_TOPK = 2

rep_model, rep_decoder, rep_activations, rep_losses, rep_xhat, rep_loss = train_sae(
    X,
    hidden_dim=REP_HIDDEN,
    topk=REP_TOPK,
    epochs=1000,
    lr=1e-2,
    seed=9423,
)

rep_features, rep_summary = summarize_features(rep_decoder, rep_activations)
rep_summary["hidden_dim"] = REP_HIDDEN
rep_summary["topk"] = REP_TOPK
rep_summary["final_loss"] = rep_loss

print(rep_summary)
rep_features

## 7. Figure 1 — SAE dictionary features

Rows are learned decoder features. Columns are residue classes mod30.


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
im = ax.imshow(rep_decoder, aspect="auto", interpolation="nearest")
ax.set_title(f"SAE Dictionary Features (hidden={REP_HIDDEN}, top-k={REP_TOPK})")
ax.set_xlabel("Residue mod 30")
ax.set_ylabel("Feature")
ax.set_xticks(range(MOD))
ax.set_yticks(range(REP_HIDDEN))

for r in VALID_LANES_MOD30:
    ax.axvline(r, linewidth=0.8, alpha=0.35)

fig.colorbar(im, ax=ax, label="decoder weight")
fig.tight_layout()
save_svg(fig, "sae_dictionary_features")
plt.show()

## 8. Figure 2 — Lane coverage sweep

Coverage is the fraction of the 8 valid lanes represented by at least one feature peak.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for topk in topks:
    subset = df_summary[df_summary["topk"] == topk].sort_values("hidden_dim")
    ax.plot(subset["hidden_dim"], subset["coverage"], marker="o", label=f"top-k={topk}")

ax.axhline(1.0, linestyle="--", alpha=0.5, label="full 8-lane coverage")
ax.set_title("SAE Lane Coverage vs Dictionary Size")
ax.set_xlabel("Dictionary size")
ax.set_ylabel("Lane coverage")
ax.set_ylim(-0.02, 1.05)
ax.legend()
fig.tight_layout()
save_svg(fig, "sae_lane_coverage")
plt.show()

## 9. Figure 3 — Dead and redundant features

Dilution appears when dictionary capacity turns into unused or duplicate features rather than new lane coverage.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
width = 0.22
x = np.arange(len(dictionary_sizes))

for idx, topk in enumerate(topks):
    subset = df_summary[df_summary["topk"] == topk].sort_values("hidden_dim")
    dilution = subset["dead_features"].to_numpy() + subset["redundant_valid_features"].to_numpy()
    ax.bar(x + (idx - 1) * width, dilution, width=width, label=f"top-k={topk}")

ax.set_xticks(x)
ax.set_xticklabels(dictionary_sizes)
ax.set_title("SAE Dilution: Dead + Redundant Features")
ax.set_xlabel("Dictionary size")
ax.set_ylabel("Feature count")
ax.legend()
fig.tight_layout()
save_svg(fig, "sae_dead_redundant_features")
plt.show()

## 10. NMF reference benchmark

Notebook 03 showed compact NMF recovery. Here we recompute a minimal NMF reference on the same matrix for a direct comparison.


In [ ]:
nmf_model = NMF(
    n_components=8,
    init="nndsvda",
    random_state=9423,
    max_iter=2000,
)
W_nmf = nmf_model.fit_transform(X)
H_nmf = nmf_model.components_
Xhat_nmf = W_nmf @ H_nmf
nmf_loss = mean_squared_error(X, Xhat_nmf)

nmf_feature_df, nmf_summary = summarize_features(H_nmf, W_nmf)
nmf_summary["final_loss"] = nmf_loss
nmf_summary

## 11. Figure 4 — SAE vs NMF comparison

The comparison is intentionally simple: coverage and lane-mass alignment.


In [ ]:
comparison = pd.DataFrame([
    {
        "model": "NMF (k=8)",
        "coverage": nmf_summary["coverage"],
        "mean_lane_mass_ratio": nmf_summary["mean_lane_mass_ratio"],
    },
    {
        "model": f"SAE (h={REP_HIDDEN}, top-k={REP_TOPK})",
        "coverage": rep_summary["coverage"],
        "mean_lane_mass_ratio": rep_summary["mean_lane_mass_ratio"],
    },
])

fig, ax = plt.subplots(figsize=(7, 5))
x = np.arange(len(comparison))
width = 0.35
ax.bar(x - width/2, comparison["coverage"], width=width, label="coverage")
ax.bar(x + width/2, comparison["mean_lane_mass_ratio"], width=width, label="mean lane mass")
ax.set_xticks(x)
ax.set_xticklabels(comparison["model"], rotation=15, ha="right")
ax.set_ylim(0, 1.05)
ax.set_title("NMF vs SAE: Residue-Manifold Recovery")
ax.set_ylabel("Score")
ax.legend()
fig.tight_layout()
save_svg(fig, "sae_nmf_comparison")
plt.show()

comparison

## 12. Save data outputs


In [ ]:
df_summary.to_csv("data/sae_dilution_summary.csv", index=False)
df_features.to_csv("data/sae_feature_summary.csv", index=False)
rep_features.to_csv("data/sae_representative_feature_summary.csv", index=False)
comparison.to_csv("data/sae_nmf_comparison.csv", index=False)

print("Saved data outputs:")
for path in [
    "data/sae_dilution_summary.csv",
    "data/sae_feature_summary.csv",
    "data/sae_representative_feature_summary.csv",
    "data/sae_nmf_comparison.csv",
]:
    print("-", path)

## 13. Interpretation

Notebook 04 supports a careful claim:

> In this controlled residue-manifold setting, a top-k SAE configuration can allocate capacity into redundant, dead, or diffuse features. NMF provides a compact baseline for recovering the 8-lane mod30 structure.

This does not claim that SAEs fail generally. It identifies a controlled **dilution regime** where sparsity and capacity do not automatically imply complete manifold coverage.


## 14. Optional download cell

Uncomment the last two lines to trigger a browser download in Colab.


In [ ]:
# --- Optional: Download outputs (uncomment last line to trigger) ---

import os
import zipfile

zip_name = "04_sae_dilution_outputs.zip"
folders_to_zip = ["data", "figures"]

with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as z:
    for folder in folders_to_zip:
        if os.path.exists(folder):
            for root, _, filenames in os.walk(folder):
                for filename in filenames:
                    path = os.path.join(root, filename)
                    z.write(path, arcname=path)

print(f"Prepared: {zip_name}")

# from google.colab import files
# files.download(zip_name)